In [ ]:

from typing import List, TypedDict
import time

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

load_dotenv()
True
docs = (
    PyPDFLoader("./documents/book1.pdf").load() +
    PyPDFLoader("./documents/book2.pdf").load() +
    PyPDFLoader("./documents/book3.pdf").load()
)
len(docs)
2123
# 2) Chunk
chunks = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=150).split_documents(docs)

# 3) Clean text to avoid UnicodeEncodeError (surrogates from PDF extraction)
for d in chunks:
    d.page_content = d.page_content.encode("utf-8", "ignore").decode("utf-8", "ignore")
len(chunks)
6396
# 3) Index (fresh collection each run)
embeddings = OpenAIEmbeddings(model='text-embedding-3-large')
vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k':4})
# 4) LLM + prompt
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
class State(TypedDict):
    question: str
    docs: List[Document]
    answer: str
def retrieve(state):
    q = state["question"]
    return {"docs": retriever.invoke(q)}
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Answer only from the context. If not in context, say you don't know."),
        ("human", "Question: {question}\n\nContext:\n{context}"),
    ]
)
def generate(state):
    context = "\n\n".join(d.page_content for d in state["docs"])
    out = (prompt | llm).invoke({"question": state["question"], "context": context})
    return {"answer": out.content}
g = StateGraph(State)
g.add_node("retrieve", retrieve)
g.add_node("generate", generate)
g.add_edge(START, "retrieve")
g.add_edge("retrieve", "generate")
g.add_edge("generate", END)
app = g.compile()

app

# 5) Run
res = app.invoke({"question": "WHat is a transformer in deep learning.", "docs": [], "answer": ""})
print(res["answer"])
A transformer in deep learning is a type of model architecture that is particularly effective for processing sequential data, such as text. It utilizes mechanisms called self-attention and feedforward neural networks to weigh the importance of different parts of the input data, allowing it to capture long-range dependencies and relationships within the data. Unlike traditional recurrent neural networks (RNNs), transformers do not process data sequentially, which enables them to be more parallelizable and efficient in training. This architecture has become foundational in natural language processing tasks and has led to significant advancements in the field.
print(res['docs'][0].page_content)
print('*'*100)
print(res['docs'][1].page_content)
print('*'*100)
print(res['docs'][2].page_content)
print('*'*100)
print(res['docs'][3].page_content)
ducingrepresentationsthatareexpressedintermsofother,simplerrepresentations.
Deeplearningallowsthecomputertobuildcomplexconceptsoutofsimplercon-
cepts.Figureshowshowadeeplearningsystemcanrepresenttheconceptof1.2
animageofapersonbycombiningsimplerconcepts,suchascornersandcontours,
whichareinturndeﬁnedintermsofedges.
Thequintessentialexampleofadeeplearningmodelisthefeedforwarddeep
networkormultilayerperceptron(MLP).Amultilayerperceptronisjusta
mathematicalfunctionmappingsomesetofinputvaluestooutputvalues.The
functionisformedbycomposingmanysimplerfunctions.Wecanthinkofeach
applicationofadiﬀerentmathematicalfunctionasprovidinganewrepresentation
oftheinput.
Theideaoflearningtherightrepresentationforthedataprovidesoneperspec-
tiveondeeplearning.Anotherperspectiveondeeplearningisthatdepthallowsthe
computertolearnamulti-stepcomputerprogram.Eachlayeroftherepresentation
****************************************************************************************************
6 A convolution is a mathematical operation that slides one function over another and measures the integral of
their pointwise multiplica
tion. It has deep connections with the Fourier transform and the Laplace transform,
and is heavily used in signal processing. Convolutional layers actually use cross-correlations, which are very
similar to convolutions (see http://goo.gl/HAfxXd for more details).
and Patrick Haffner, which introduced the famous L eNe t-5 architecture, widely used
to recognize handwritten check numbers. This architecture has some building blocks
that you already know, such as fully connected layers and sigmoid activation func‐
tions, but it also introduces two new building blocks: convolutional layers  and pooling
layers. Let’s look at them now.
Why not simply use a regular deep neural network with fully con‐
****************************************************************************************************
network.Thediﬀerenceisthatinthecaseofdatasetaugmentation,thenetworkis
explicitlytrainedtocorrectlyclassifydistinctinputsthatwerecreatedbyapplying
morethananinﬁnitesimalamountofthesetransformations.Tangentpropagation
doesnotrequireexplicitlyvisitinganewinputpoint.Instead,itanalytically
regularizesthemodeltoresistperturbationinthedirectionscorrespondingto
the speciﬁed transformation.While thisanalytical approach isintellectually
elegant,ithastwomajordrawbacks.First,itonlyregularizesthemodeltoresist
inﬁnitesimalperturbation.Explicitdatasetaugmentationconfersresistanceto
largerperturbations.Second,theinﬁnitesimalapproachposesdiﬃcultiesformodels
basedonrectiﬁedlinearunits.Thesemodelscanonlyshrinktheirderivatives
byturningunitsoﬀorshrinkingtheirweights.Theyarenotabletoshrinktheir
derivativesbysaturatingatahighvaluewithlargeweights,assigmoidortanh
****************************************************************************************************
training and visualizing, 167-169
decoder, 412
deconvolutional layer, 
376
deep autoencoders (see stacked autoencoders)
deep belief networks (DBNs), 13, 519-521
Deep Learning, 437
(see also Reinforcement Learning; Tensor‐
Flow)
about, xiii, xvi
libraries, 230-231
deep neural networks (DNNs), 261, 275-312
(see also Multi-Layer Perceptrons (MLP))
faster optimizers for, 293-302
regularization, 302-310
reusing pretrained layers, 286-293
training guidelines overview, 310
training with TensorFlow, 265-270
training with TF .Learn, 264
unstable gradients, 276
vanishing and exploding gradients, 275-286
Deep Q-Learning, 460-469
Ms. Pac Man example, 460-469
deep Q-network, 460
deep RNNs, 396-400
applying dropout, 399
distributing across multiple GPUs, 397
long sequence difficulties, 400
truncated backpropagation through time,
400
DeepMind, 14, 253, 437, 460
degrees of freedom, 27, 126
 
 
 
 